# 00 — Data Validation

Validates schema, ranges, keys, provenance, collection coverage, and submission readiness. Raw inputs are never modified.

In [1]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
fingerprint_paths = [PROJECT_ROOT / "config" / "config.yaml", project_path("shops")]
for key in ("products", "reviews"):
    fingerprint_paths.extend(sorted(project_path(key).glob("shop_*.json")))
seller_candidate = project_path("seller_metrics")
for candidate in (seller_candidate, seller_candidate.with_suffix(".xlsx")):
    if candidate.exists(): fingerprint_paths.append(candidate)
digest = hashlib.sha256()
for path in fingerprint_paths:
    digest.update(str(path.relative_to(PROJECT_ROOT)).encode("utf-8")); digest.update(path.read_bytes())
INPUT_FINGERPRINT = digest.hexdigest()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_FINGERPRINT:", INPUT_FINGERPRINT)

PROJECT_ROOT: /app/workspace/shopee_dikw
INPUT_FINGERPRINT: 4492150003866301d600af9db4ec8673f81d03d7f250aa4aa1e04b912fd24f64


In [2]:
SHOP_REQUIRED = ["Shop_ID", "Shop_Name", "Shop_type", "Years_Active", "Is_Online_Now", "Total_Products", "Follower_Count_k", "Rating_Average", "Total_Ratings_k", "Chat_Response_Rate_pct", "Has_Voucher", "Target_Label", "Time_Collected"]
PRODUCT_REQUIRED = ["Shop_ID", "product_id", "product_name", "product_details", "description_text", "Review_stars", "units_sold", "product_url", "Time_Collected", "Data_Source"]
REVIEW_REQUIRED = ["Shop_ID", "product_id", "review_id", "user_name", "rating", "review_time", "review_text", "has_image", "source_url", "Time_Collected", "Data_Source", "Verification_Status"]
SELLER_REQUIRED = ["Shop_ID", "Conversion_Rate_pct", "Source", "Collection_Status"]

def load_json_folder(folder, pattern, columns):
    records = []
    for path in sorted(folder.glob(pattern)):
        with path.open(encoding="utf-8") as handle:
            payload = json.load(handle)
        if not isinstance(payload, list):
            raise ValueError(f"{path}: top-level JSON must be a list")
        records.extend(payload)
    return pd.DataFrame(records) if records else pd.DataFrame(columns=columns)

def load_seller_metrics(path):
    candidates = [path, path.with_suffix(".xlsx")]
    source = next((p for p in candidates if p.exists()), None)
    if source is None:
        return pd.DataFrame(columns=SELLER_REQUIRED), None
    frame = pd.read_excel(source, dtype={"Shop_ID": str}) if source.suffix == ".xlsx" else pd.read_csv(source, dtype={"Shop_ID": str})
    return frame, source

shops = pd.read_excel(project_path("shops"), dtype={"Shop_ID": str})
products = load_json_folder(project_path("products"), "shop_*_products.json", PRODUCT_REQUIRED)
reviews = load_json_folder(project_path("reviews"), "shop_*_reviews.json", REVIEW_REQUIRED)
seller_metrics, seller_source = load_seller_metrics(project_path("seller_metrics"))

# Canonicalize legacy/manual collection fields without inventing business values.
manifest_path = PROJECT_ROOT / "src" / "crawl_data" / "product_review_mapping.json"
if manifest_path.exists():
    manifest = pd.DataFrame(json.loads(manifest_path.read_text(encoding="utf-8")))
    if {"product_id", "product_url"} <= set(manifest):
        url_map = manifest.drop_duplicates("product_id").set_index("product_id")["product_url"]
        if "product_url" not in products: products["product_url"] = products["product_id"].map(url_map)
        else: products["product_url"] = products["product_url"].fillna(products["product_id"].map(url_map))
for column in PRODUCT_REQUIRED:
    if column not in products: products[column] = np.nan
if "product_details" in products:
    detail_url = products["product_details"].map(lambda value: value.get("source_product_url") if isinstance(value, dict) else None)
    detail_date = products["product_details"].map(lambda value: value.get("collection_date") if isinstance(value, dict) else None)
    sold_display = products["product_details"].map(lambda value: value.get("public_sold_display") if isinstance(value, dict) else None)

    def parse_public_sold_lower_bound(value):
        """Parse Shopee's public rounded sold display as an observed lower bound."""
        if pd.isna(value):
            return np.nan
        match = __import__("re").search(r"([0-9]+(?:[.,][0-9]+)?)\s*([kKmM]?)", str(value))
        if not match:
            return np.nan
        number = float(match.group(1).replace(",", "."))
        multiplier = {"": 1, "k": 1_000, "m": 1_000_000}[match.group(2).lower()]
        return number * multiplier

    products["product_url"] = products["product_url"].fillna(detail_url)
    products["Time_Collected"] = products["Time_Collected"].fillna(detail_date)
    products["units_sold"] = pd.to_numeric(products["units_sold"], errors="coerce").fillna(sold_display.map(parse_public_sold_lower_bound))
    products["units_sold_display"] = sold_display
    products["units_sold_semantics"] = products["units_sold"].notna().map({True: "public_display_lower_bound", False: None})
products["Data_Source"] = products["Data_Source"].fillna(products["product_url"])
for column in REVIEW_REQUIRED:
    if column not in reviews: reviews[column] = np.nan
if "collection_date" in reviews: reviews["Time_Collected"] = reviews["Time_Collected"].fillna(reviews["collection_date"])
# Reviews were collected from each product's public review endpoint/page.  Link
# provenance through product_id in memory so the immutable raw JSON is not
# rewritten merely to duplicate product-level collection metadata.
product_provenance = products.drop_duplicates("product_id").set_index("product_id")
review_source = reviews["product_id"].map(product_provenance["product_url"])
review_collected = reviews["product_id"].map(product_provenance["Time_Collected"])
reviews["source_url"] = reviews["source_url"].fillna(review_source)
reviews["Time_Collected"] = reviews["Time_Collected"].fillna(review_collected)
reviews["Data_Source"] = reviews["Data_Source"].fillna(reviews["source_url"])
linked = reviews["source_url"].notna() & reviews["Time_Collected"].notna()
reviews.loc[linked & reviews["Verification_Status"].isna(), "Verification_Status"] = "linked_to_product_provenance"
print({"shops": len(shops), "products": len(products), "reviews": len(reviews), "seller_metrics": len(seller_metrics)})

{'shops': 15, 'products': 100, 'reviews': 750, 'seller_metrics': 0}


In [3]:
checks = []
def check(name, severity, invalid_count, description):
    invalid_count = int(invalid_count)
    checks.append({"check_name": name, "severity": severity, "status": "PASS" if invalid_count == 0 else severity, "error_count": invalid_count, "description": description})

def missing_count(frame, required):
    return len(set(required) - set(frame.columns))

for name, frame, required in [("shops", shops, SHOP_REQUIRED), ("products", products, PRODUCT_REQUIRED), ("reviews", reviews, REVIEW_REQUIRED), ("seller_metrics", seller_metrics, SELLER_REQUIRED)]:
    check(f"{name}_required_columns", "FAIL", missing_count(frame, required), f"Required schema: {required}")

if not missing_count(shops, SHOP_REQUIRED):
    check("unique_Shop_ID", "FAIL", shops["Shop_ID"].duplicated().sum(), "Shop_ID must be unique")
    check("shop_rating_range", "FAIL", (~pd.to_numeric(shops["Rating_Average"], errors="coerce").between(0, 5)).sum(), "Rating_Average in [0,5]")
    check("response_rate_range", "FAIL", (~pd.to_numeric(shops["Chat_Response_Rate_pct"], errors="coerce").between(0, 100)).sum(), "Chat response in [0,100]")
    check("shop_dates", "FAIL", pd.to_datetime(shops["Time_Collected"], errors="coerce").isna().sum(), "Time_Collected must be a date")
if not missing_count(products, PRODUCT_REQUIRED):
    check("unique_product_id", "FAIL", products["product_id"].duplicated().sum(), "product_id must be globally unique")
    product_rating = pd.to_numeric(products["Review_stars"], errors="coerce")
    product_sales = pd.to_numeric(products["units_sold"], errors="coerce")
    check("product_rating_range", "FAIL", (product_rating.notna() & ~product_rating.between(0, 5)).sum(), "Observed Review_stars in [0,5]")
    check("product_rating_missing", "WARNING", product_rating.isna().sum(), "Missing rating remains null")
    check("product_units_sold_negative", "FAIL", (product_sales.notna() & (product_sales < 0)).sum(), "Observed units_sold must be non-negative")
    check("product_units_sold_missing", "WARNING", product_sales.isna().sum(), "Missing sales remains null; never substitute ratings")
    check("product_provenance", "WARNING", products[["product_url", "Time_Collected", "Data_Source"]].isna().any(axis=1).sum(), "Products should have URL, date and source")
if not missing_count(reviews, REVIEW_REQUIRED):
    check("unique_review_id", "FAIL", reviews["review_id"].duplicated().sum(), "review_id must be unique")
    review_rating = pd.to_numeric(reviews["rating"], errors="coerce")
    review_image = pd.to_numeric(reviews["has_image"], errors="coerce")
    check("review_rating_range", "FAIL", (review_rating.notna() & ~review_rating.between(1, 5)).sum(), "Observed rating in [1,5]")
    check("review_rating_missing", "WARNING", review_rating.isna().sum(), "Unverified rating remains null")
    check("review_has_image", "FAIL", (review_image.notna() & ~review_image.isin([0, 1])).sum(), "Observed has_image must be binary")
    check("review_has_image_missing", "WARNING", review_image.isna().sum(), "Unverified image flag remains null")
    check("review_text", "WARNING", reviews["review_text"].fillna("").astype(str).str.strip().eq("").sum(), "Rating-only reviews remain raw but are excluded from text analysis")
    check("review_provenance", "WARNING", reviews[["source_url", "Time_Collected", "Data_Source", "Verification_Status"]].isna().any(axis=1).sum(), "Reviews should have URL, collection date, source and verification status")
if not missing_count(seller_metrics, SELLER_REQUIRED):
    seller_keys = ["Shop_ID", "product_id"] if "product_id" in seller_metrics else ["Shop_ID"]
    check("seller_key_unique", "FAIL", seller_metrics.duplicated(seller_keys).sum(), f"Seller export key must be unique: {seller_keys}")
    conversion = pd.to_numeric(seller_metrics["Conversion_Rate_pct"], errors="coerce")
    check("conversion_range", "FAIL", (conversion.notna() & ~conversion.between(0,100)).sum(), "Observed conversion rate in [0,100]")
    check("conversion_missing", "WARNING", conversion.isna().sum(), "Missing Seller Centre conversion remains null")

if {"Shop_ID"} <= set(shops) and {"Shop_ID", "product_id"} <= set(products) and {"Shop_ID", "product_id"} <= set(reviews):
    check("product_shop_fk", "FAIL", len(set(products["Shop_ID"]) - set(shops["Shop_ID"])), "Every product shop exists")
    check("review_shop_fk", "FAIL", len(set(reviews["Shop_ID"]) - set(shops["Shop_ID"])), "Every review shop exists")
    check("review_product_fk", "FAIL", len(set(reviews["product_id"]) - set(products["product_id"])), "Every review product exists")
    product_shop = products.set_index("product_id")["Shop_ID"].to_dict()
    check("review_product_shop_match", "FAIL", reviews.apply(lambda r: product_shop.get(r["product_id"]) not in (None, r["Shop_ID"]), axis=1).sum(), "Review Shop_ID agrees with its product")

dq = CONFIG["data_quality"]
check("minimum_shops", "WARNING", max(0, dq["minimum_shops"] - shops["Shop_ID"].nunique()), f"At least {dq['minimum_shops']} real shops")
product_counts = products.groupby("Shop_ID").size().reindex(shops["Shop_ID"], fill_value=0)
reviews_with_text = reviews.loc[reviews["review_text"].fillna("").astype(str).str.strip().ne("")]
review_counts = reviews_with_text.groupby(["Shop_ID", "product_id"]).size().reindex(pd.MultiIndex.from_frame(products[["Shop_ID", "product_id"]]), fill_value=0)
check("products_per_shop", "WARNING", ((product_counts < dq["minimum_products_per_shop"]) | (product_counts > dq["maximum_products_per_shop"])).sum(), "Each shop needs 5–10 products")
check("reviews_per_product", "WARNING", (review_counts < dq["minimum_text_reviews_per_product"]).sum(), "Each product needs at least five text reviews")
observed_conversion = pd.to_numeric(seller_metrics.get("Conversion_Rate_pct", pd.Series(dtype=float)),errors="coerce")
check("seller_metrics_present", "WARNING", int(seller_metrics.empty or not observed_conversion.notna().any()), "Observed authorized Seller Centre conversion is required")

coverage = pd.DataFrame({"Shop_ID": shops["Shop_ID"], "Product_Count": shops["Shop_ID"].map(product_counts).fillna(0).astype(int)})
coverage["Review_Count"] = coverage["Shop_ID"].map(reviews_with_text.groupby("Shop_ID").size()).fillna(0).astype(int)
seller_observed = seller_metrics.loc[observed_conversion.notna()] if not seller_metrics.empty else seller_metrics
coverage["Products_With_Conversion"] = coverage["Shop_ID"].map(seller_observed.groupby("Shop_ID").size() if not seller_observed.empty else {}).fillna(0).astype(int)
report = pd.DataFrame(checks)
report.to_csv(OUTPUT / "reports" / "data_validation_report.csv", index=False)
coverage.to_csv(OUTPUT / "reports" / "data_coverage_by_shop.csv", index=False)
display(report); display(coverage)
critical = report.query("status == 'FAIL'")
if not critical.empty:
    raise ValueError("Critical validation failed: " + ", ".join(critical["check_name"]))

,check_name,severity,status,error_count,description
0,shops_required_columns,FAIL,PASS,0,"Required schema: ['Shop_ID', 'Shop_Name', 'Sho..."
1,products_required_columns,FAIL,PASS,0,"Required schema: ['Shop_ID', 'product_id', 'pr..."
2,reviews_required_columns,FAIL,PASS,0,"Required schema: ['Shop_ID', 'product_id', 're..."
3,seller_metrics_required_columns,FAIL,PASS,0,"Required schema: ['Shop_ID', 'Conversion_Rate_..."
4,unique_Shop_ID,FAIL,PASS,0,Shop_ID must be unique
5,shop_rating_range,FAIL,PASS,0,"Rating_Average in [0,5]"
6,response_rate_range,FAIL,PASS,0,"Chat response in [0,100]"
7,shop_dates,FAIL,PASS,0,Time_Collected must be a date
8,unique_product_id,FAIL,PASS,0,product_id must be globally unique
9,product_rating_range,FAIL,PASS,0,"Observed Review_stars in [0,5]"


,Shop_ID,Product_Count,Review_Count,Products_With_Conversion
0,01_001,10,31,0
1,01_002,10,50,0
2,01_003,10,50,0
3,01_004,10,50,0
4,01_005,10,50,0
5,01_006,5,50,0
6,01_007,5,50,0
7,01_008,5,50,0
8,01_009,5,50,0
9,01_010,5,50,0


In [4]:
import re
def exists(rel): return (PROJECT_ROOT / rel).exists()
metrics_path = OUTPUT / "reports" / "classification_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {}
regression_metrics_path = OUTPUT / "reports" / "regression_metrics.json"
regression_metrics = json.loads(regression_metrics_path.read_text(encoding="utf-8")) if regression_metrics_path.exists() else {}
wisdom_path = OUTPUT / "reports" / "wisdom_facts.json"
wisdom = json.loads(wisdom_path.read_text(encoding="utf-8")) if wisdom_path.exists() else {}
mapping = CONFIG["clustering"].get("authenticity_mapping") or {}
pbix = next((path for path in (PROJECT_ROOT / "powerbi").glob("*.pbix")), None)
pbix_evidence_path = PROJECT_ROOT / "powerbi" / "verification.json"
pbix_evidence = json.loads(pbix_evidence_path.read_text(encoding="utf-8")) if pbix_evidence_path.exists() else {}
required_pages = {"Tổng quan shop", "Sản phẩm và review", "Uy tín và chuyển đổi"}
powerbi_verified = bool(pbix and required_pages <= set(pbix_evidence.get("pages", [])) and pbix_evidence.get("refresh_passed") is True and pbix_evidence.get("cross_filter_passed") is True and str(pbix_evidence.get("verified_by", "")).strip() and str(pbix_evidence.get("verified_at", "")).strip() and pbix_evidence.get("input_fingerprint") == INPUT_FINGERPRINT)
metadata_path = PROJECT_ROOT / "submission" / "submission_metadata.yaml"
metadata = yaml.safe_load(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
links_verified = bool(str(metadata.get("colab_url", "")).startswith("https://colab.research.google.com") and str(metadata.get("drive_url", "")).startswith("https://drive.google.com") and metadata.get("access_verified") is True and str(metadata.get("verified_by", "")).strip() and str(metadata.get("verified_at", "")).strip() and metadata.get("input_fingerprint") == INPUT_FINGERPRINT)
report_path = PROJECT_ROOT / "report" / "Final_Report.pdf"
report_pages = len(re.findall(rb"/Type\s*/Page(?!s)\b", report_path.read_bytes())) if report_path.exists() else 0
coverage_ready = bool(len(shops) >= dq["minimum_shops"] and product_counts.between(dq["minimum_products_per_shop"], dq["maximum_products_per_shop"]).all() and (review_counts >= dq["minimum_text_reviews_per_product"]).all())
products_per_shop_ready = bool(len(product_counts) and product_counts.between(dq["minimum_products_per_shop"], dq["maximum_products_per_shop"]).all())
reviews_per_product_ready = bool(len(review_counts) and (review_counts >= dq["minimum_text_reviews_per_product"]).all())
auth_metadata_path = OUTPUT / "reports" / "authenticity_metadata.json"
auth_metadata = json.loads(auth_metadata_path.read_text(encoding="utf-8")) if auth_metadata_path.exists() else {}
authenticity_ready = bool(mapping and exists("output/reports/Shop_Authenticity_Report.csv") and auth_metadata.get("input_fingerprint") == INPUT_FINGERPRINT and auth_metadata.get("authenticity_mapping") == {str(k):float(v) for k,v in mapping.items()})
items = {
    "THREE_DATA_TYPES": bool(len(shops) and len(products) and len(reviews)),
    "MINIMUM_REAL_SHOPS": shops["Shop_ID"].nunique() >= dq["minimum_shops"],
    "PRODUCTS_PER_SHOP": products_per_shop_ready,
    "REVIEWS_PER_PRODUCT": reviews_per_product_ready,
    "REVIEW_METADATA_COMPLETE": bool(reviews["rating"].notna().all() and reviews["has_image"].notna().all()),
    "PROVENANCE_COMPLETE": bool(products[["product_url","Time_Collected","Data_Source"]].notna().all(axis=1).all() and reviews[["source_url","Time_Collected","Data_Source","Verification_Status"]].notna().all(axis=1).all()),
    "SELLER_METRICS": bool(not seller_metrics.empty and observed_conversion.notna().all()),
    "AUTHENTICITY_MAPPING": bool(mapping) and set(map(float,mapping.values())) == {0.0,0.5,1.0},
    "AUTHENTICITY_REPORT": authenticity_ready,
    "CLASSIFICATION_RESULT": exists("output/reports/Classification_Result.csv") and metrics.get("input_fingerprint") == INPUT_FINGERPRINT and metrics.get("valid_ground_truth_evaluation") is True,
    "CLASSIFICATION_ACCURACY": metrics.get("input_fingerprint") == INPUT_FINGERPRINT and metrics.get("valid_ground_truth_evaluation") is True and float(metrics.get("accuracy",0)) > float(CONFIG["classification"]["accuracy_target"]),
    "CLASSIFICATION_MACRO_F1": metrics.get("input_fingerprint") == INPUT_FINGERPRINT and metrics.get("valid_ground_truth_evaluation") is True and pd.notna(metrics.get("macro_f1")),
    "REGRESSION_RESULT": exists("output/reports/Regression_Result.csv") and regression_metrics.get("input_fingerprint") == INPUT_FINGERPRINT,
    "WISDOM_FACTS": wisdom.get("input_fingerprint") == INPUT_FINGERPRINT,
    "POWER_BI_VERIFIED": powerbi_verified,
    "FINAL_REPORT_10_PAGES": report_pages >= 10,
    "COLAB_DRIVE_ACCESS": links_verified,
}
items = {name:bool(value) for name,value in items.items()}
text = "\n".join(f"{name:<32} {'PASS' if value else 'FAIL'}" for name, value in items.items())
text += f"\n\nCOUNTS: shops={shops['Shop_ID'].nunique()}, products={products['product_id'].nunique()}, reviews={len(reviews)}, text_reviews={len(reviews_with_text)}\nFINAL_REPORT_PAGES: {report_pages}\n\nOVERALL: {'READY FOR SUBMISSION' if all(items.values()) else 'NOT READY FOR SUBMISSION'}\n"
(OUTPUT / "reports" / "submission_check.txt").write_text(text, encoding="utf-8")
(OUTPUT / "reports" / "submission_check.json").write_text(json.dumps({"checks":items,"overall_ready":all(items.values())},ensure_ascii=False,indent=2),encoding="utf-8")
print(text)

THREE_DATA_TYPES                 PASS
MINIMUM_REAL_SHOPS               PASS
PRODUCTS_PER_SHOP                PASS
REVIEWS_PER_PRODUCT              FAIL
REVIEW_METADATA_COMPLETE         PASS
PROVENANCE_COMPLETE              PASS
SELLER_METRICS                   FAIL
AUTHENTICITY_MAPPING             PASS
AUTHENTICITY_REPORT              FAIL
CLASSIFICATION_RESULT            FAIL
CLASSIFICATION_ACCURACY          FAIL
CLASSIFICATION_MACRO_F1          FAIL
REGRESSION_RESULT                FAIL
WISDOM_FACTS                     FAIL
POWER_BI_VERIFIED                FAIL
FINAL_REPORT_10_PAGES            FAIL
COLAB_DRIVE_ACCESS               FAIL

COUNTS: shops=15, products=100, reviews=750, text_reviews=731
FINAL_REPORT_PAGES: 0

OVERALL: NOT READY FOR SUBMISSION

